In [1]:
import os
import git
from pathlib import Path
from typing import List
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import plotly.graph_objects as go
from IPython.display import clear_output
import scipy
from scipy import stats
from scipy.spatial import ConvexHull
import pylustrator
from scipy.spatial import Delaunay
from scipy.spatial import distance
from sklearn.decomposition import PCA
from tqdm import tqdm
import warnings

warnings.simplefilter("ignore", RuntimeWarning)
ROOT_DIR =  Path(git.Repo('.', search_parent_directories=True).working_tree_dir)
save_path_with_hull = Path(os.path.join(ROOT_DIR, 'publication', 'paper', 'CSVs', 'final_results_audio_with_hull.pickle'))
save_path_without_hull = Path(os.path.join(ROOT_DIR, 'publication', 'paper', 'CSVs', 'final_results_audio.csv'))
plots_path = os.path.join(ROOT_DIR, "publication", "paper", "draft_plots")

In [2]:
def variance_prior(r, eta, scale=1):
    beta = (eta+1.5)/r
    var_prior = scale * scipy.special.gamma(beta + 1/r)/scipy.special.gamma(beta)
    return var_prior

def kurtosis_prior(r, eta, scale=1, fisher=True):
    beta = (eta+1.5)/r
    kurtosis = scale*3*scipy.special.gamma(beta + 2/r)*scipy.special.gamma(beta)/scipy.special.gamma(beta+1/r)**2 # DOUBLE CHECK
    if fisher:
        return kurtosis - 3
    else:
        return kurtosis 

def find_master_dfs(root_dir: str) -> List[str]:
    root_path = Path(root_dir)
    if not root_path.exists():
        raise FileNotFoundError(f"Directory not found: {root_dir}")

    master_df_paths = []
    for current_dir, _, files in os.walk(root_path):
        if 'master_df.csv' in files:
            master_df_path = Path(os.path.join(current_dir, 'master_df.csv'))
            master_df_paths.append(str(master_df_path.absolute()))
    return master_df_paths

In [3]:
def add_hull(master_df, rEtaKsstats_dict, GROUP='group', debug=False):

    master_df_copy = master_df.copy()
    master_df_copy = master_df.set_index(GROUP)
    groups = master_df_copy.index
    master_df_copy["hull"] = ""

    for group in groups:
        if master_df_copy.loc[group, "total_samples"] < 10:
            master_df_copy.loc[group, "hull"] = np.nan
           
        else:
            drop_keys =list(rEtaKsstats_dict[group].keys())[-3:]
            if debug:
                print(drop_keys)
            pre_optimization = pd.DataFrame(rEtaKsstats_dict[group]).drop(drop_keys, axis = 1 )
            optimization = pd.DataFrame(rEtaKsstats_dict[group])[drop_keys]
            optimization = optimization.rename(columns = {"r_optimize": "r", "eta_optimize": "eta", drop_keys[-1]: "ksstat"})
            optimization = optimization.dropna()
            full_df = pre_optimization.merge(optimization, on=["r", "eta"], how="outer")
            full_df = full_df.set_index(["r", "eta"])
            full_df["ksstat"] = full_df.min(axis=1)
            full_df = full_df.reset_index()
            full_df = full_df[["r", "eta", "ksstat"]]
            full_df["1/beta"] = full_df["r"]/(full_df["eta"] + 1.5)
            full_df["log_beta"] = np.log10((full_df["eta"] + 1.5) / full_df["r"])
            MULT = 1.2
            cutoff = max(min(full_df["ksstat"]) * MULT, 0.01)
            filtered_df = full_df[full_df["ksstat"] < cutoff]
            points = np.column_stack((filtered_df["r"], filtered_df["1/beta"])) + stats.norm.rvs(size=(len(filtered_df), 2)) * 0.001  # Adding small noise for convex hull computation
            #points = np.column_stack((np.log10(filtered_df["r"]), filtered_df["log_beta"])) + stats.norm.rvs(size=(len(filtered_df), 2)) * 0.001  # Adding small noise for convex hull computation
            if len(points) < 3:
                hull=np.nan
                master_df_copy.loc[group, "intersect_roi"] = -1
            else:
                hull = ConvexHull(points)

                if np.any(filtered_df["eta"] > 0) and np.any(filtered_df["eta"] < 0):
                    intersect_roi = 1
                else:
                    intersect_roi = 0
                master_df_copy.loc[group, "intersect_roi"] = intersect_roi
            master_df_copy.loc[group, "hull"] = hull

    return master_df_copy.reset_index()

In [4]:
def process_rEtaKsstats_dict(master_df_original, rEtaKsstats_dict, GROUP='group', debug=False):
    master_df = master_df_original.copy()
    master_df = master_df_original.set_index(GROUP)
    groups = master_df.index
    master_df["hull"] = ""
    for group in groups:
        if master_df.loc[group, "total_samples"] < 10:
            master_df.loc[group, "hull"] = np.nan
        else:
            dict_keys = list(rEtaKsstats_dict[group].keys())
            if "r_optimize" in dict_keys and "eta_optimize" in dict_keys:
                drop_keys = ["r_optimize", "eta_optimize", dict_keys[-1]]
                if debug:
                    print(drop_keys)
                pre_optimization = pd.DataFrame(rEtaKsstats_dict[group]).drop(drop_keys, axis=1)
                optimization = pd.DataFrame(rEtaKsstats_dict[group])[drop_keys]
                optimization = optimization.rename(columns={"r_optimize": "r", "eta_optimize": "eta", dict_keys[-1]: "ksstat"})
                optimization = optimization.dropna()
                full_df = pre_optimization.merge(optimization, on=["r", "eta"], how="outer")
            else:
                full_df = pd.DataFrame(rEtaKsstats_dict[group])
            full_df = full_df.set_index(["r", "eta"])
            full_df["ksstat"] = full_df.min(axis=1)
            full_df = full_df.reset_index()
            full_df = full_df[["r", "eta", "ksstat"]]
            full_df["1/beta"] = full_df["r"] / (full_df["eta"] + 1.5)
            
            MULT = 1.2
            cutoff = max(min(full_df["ksstat"]) * MULT, 0.01)
            if min(full_df["ksstat"]) * MULT > 0.01:
                uses_practical_threshold = 0
            else:
                uses_practical_threshold = 1
            master_df.loc[group, "use_practical_threshold"] = uses_practical_threshold 
            
            filtered_df = full_df[full_df["ksstat"] < cutoff]
            points = np.column_stack((filtered_df["r"], filtered_df["1/beta"])) + stats.norm.rvs(size=(len(filtered_df), 2)) * 0.001 
            if len(points) < 3:
                hull = np.nan
                hull_area = 0
                intersect_roi = 0
            else:
                hull = ConvexHull(points)
                hull_area = hull.volume
                if np.any(filtered_df["eta"] > 0) and np.any(filtered_df["eta"] < 0):
                    intersect_roi = 1
                else:
                    intersect_roi = 0
            master_df.loc[group, "hull"] = hull    
            master_df.loc[group, "hull_area"] = hull_area
            master_df.loc[group, "intersect_roi"] = intersect_roi
            master_df.loc[group, "hull_r_lower"] = filtered_df['r'].min()
            master_df.loc[group, "hull_r_upper"] = filtered_df['r'].max()
            master_df.loc[group, "hull_beta_lower"] = 1/filtered_df['1/beta'].max()
            master_df.loc[group, "hull_beta_upper"] = 1/filtered_df['1/beta'].min()
    return master_df.reset_index()

In [5]:
all_paths = find_master_dfs(os.path.join(ROOT_DIR, "results-audio", "case-studies"))
all_paths[:10]

['e:\\Research\\UCB\\Strang Lab\\Github\\hierarchical-bayesian-model-validation\\results-audio\\case-studies\\ravdess\\1e5geocomp\\fft\\female\\CSVs\\master_df.csv',
 'e:\\Research\\UCB\\Strang Lab\\Github\\hierarchical-bayesian-model-validation\\results-audio\\case-studies\\ravdess\\1e5geocomp\\fft\\male\\CSVs\\master_df.csv',
 'e:\\Research\\UCB\\Strang Lab\\Github\\hierarchical-bayesian-model-validation\\results-audio\\case-studies\\ravdess\\1e5geocomp\\fft\\normal_intensity\\CSVs\\master_df.csv',
 'e:\\Research\\UCB\\Strang Lab\\Github\\hierarchical-bayesian-model-validation\\results-audio\\case-studies\\ravdess\\1e5geocomp\\fft\\statement_1\\CSVs\\master_df.csv',
 'e:\\Research\\UCB\\Strang Lab\\Github\\hierarchical-bayesian-model-validation\\results-audio\\case-studies\\ravdess\\1e5geocomp\\fft\\statement_2\\CSVs\\master_df.csv',
 'e:\\Research\\UCB\\Strang Lab\\Github\\hierarchical-bayesian-model-validation\\results-audio\\case-studies\\ravdess\\1e5geocomp\\fft\\strong_intensity

In [7]:
relevant_cols = [
        'group', 'dataset', 'subset', 'transform', 'orientation', 'channel', 'dataset_type', 
        'obs_var', 'var_lower', 'var_upper', 
        'total_samples', 'initial_r', 'initial_eta',  'best_r', 'best_eta', 'best_scale',
        'kstest_stat_initial', 'kstest_stat_cutoff_0.05', 'kstest_stat_best', 'n_pval_0.05', 
        'obs_kurt', 'kurt_lower', 'kurt_upper', 
        'intersect_roi', 'hull', 'hull_area',
        'hull_r_lower', 'hull_r_upper', 'hull_beta_lower', 'hull_beta_upper', 
        # 'num_pass_kurt_anywhere', 'pass_kurt_anywhere', 'num_pass_kurt_intersect_hull', 'pass_kurt_intersect_hull', 'hull_kurt', 'hull_kurt_area',
        'param_gaussian', 'kstest_stat_gaussian', 'kstest_pval_gaussian', 
        'param_laplace', 'kstest_stat_laplace', 'kstest_pval_laplace', 
        'param_t', 'kstest_stat_t', 'kstest_pval_t', 'kstest_pval_gengamma', 
        'github_plot']
all_paths = find_master_dfs(os.path.join(ROOT_DIR, "results-audio", "case-studies"))
all_master_dfs = []
github_plots_path = "https://github.com/yashdave003/hierarchical-bayesian-model-validation/blob/main/results-audio/case-studies/"
for i in tqdm(range(len(all_paths))):
    path = all_paths[i]
    if 'scaleTesting' in path:
        continue
    master_df = pd.read_csv(path)
    master_df = master_df.rename(columns={master_df.columns[0]: 'group'})
    parts = Path(path).parts[-7:]
    if parts[0] == 'case-studies':
        parts = parts[1:]
    elif parts[0] == 'results-audio':
        parts = parts[2:]
    if "MRI" in path:
        dataset, slice, transform, orientation, _, _ = parts
        master_df['dataset'] = dataset
        master_df['transform'] = transform
        master_df['subset'] = slice
        master_df['channel'] = np.nan
        master_df['orientation'] = orientation
        master_df['github_plot'] = [github_plots_path+'/'.join([dataset, slice, transform, orientation, 'plots', f'compare_cdf_pdf_layer_{group}.jpg']) for group in master_df['group']]
    elif len(parts) > 6:
        dataset, subset, transform, orientation, channel, _, _ = parts
        master_df['dataset'] = dataset
        master_df['transform'] = transform
        master_df['subset'] = subset
        master_df['channel'] = channel
        master_df['orientation'] = orientation
        master_df['github_plot'] = [github_plots_path+'/'.join([dataset, subset, transform, orientation, channel, 'plots', f'compare_cdf_pdf_layer_{group}.jpg']) for group in master_df['group']]
    elif "learned" in path:
        dataset, subset, transform, _, _ = parts
        master_df['dataset'] = dataset
        master_df['transform'] = transform
        master_df['subset'] = subset
        master_df = master_df.rename(columns={'filter_group' : 'orientation'})
        master_df['channel'] = np.nan
        master_df['github_plot'] = [github_plots_path+'/'.join([dataset, subset, transform, 'plots', f'compare_cdf_pdf_layer_{group}.jpg']) for group in master_df['group']]
    else:
        dataset, size, transform, channel, _, _ = parts
        master_df['dataset'] = dataset
        master_df['transform'] = transform
        master_df['subset'] = size
        master_df['channel'] = channel
        master_df['orientation'] = np.nan
        master_df['github_plot'] = [github_plots_path+'/'.join([dataset, size, transform, channel, 'plots', f'compare_cdf_pdf_layer_{group}.jpg']) for group in master_df['group']]
    
    if dataset in ['ravdess']:
        master_df['dataset_type'] = 'speech'
    # elif dataset in ['syntheticMRI2D', 'syntheticMRI3D']:
    #     master_df['dataset_type'] = 'medical'
    # elif dataset in ['coco', 'segmentAnything', 'standardTesting']:
    #     master_df['dataset_type'] = 'natural'
    # elif dataset in ['standardTesting']:
    #     master_df['dataset_type'] = 'classical'
        GROUP = 'band'
    rEtaKsstatsDict = pd.read_pickle(path[:-18] + "cache" + os.sep + "rEtaKsstats_dict.pickle")
    try:
        master_df = process_rEtaKsstats_dict(master_df, rEtaKsstatsDict)
    except Exception as e:
        print(f"\nFAILED at i={i}")
        print(f"path: {path}")
        print(f"error: {type(e).__name__}: {e}")
        # Find exact failing group and missing columns using robust logic
        tmp = master_df.set_index('group')
        for g in tmp.index:
            if tmp.loc[g, "total_samples"] < 10:
                continue
            dfg = pd.DataFrame(rEtaKsstatsDict[g])
            dict_keys = list(rEtaKsstatsDict[g].keys())
            
            if "r_optimize" in dict_keys and "eta_optimize" in dict_keys:
                drop_keys = ["r_optimize", "eta_optimize", dict_keys[-1]]
                pre = dfg.drop(drop_keys, axis=1, errors='ignore')
                opt = dfg[drop_keys].rename(columns={
                    "r_optimize": "r",
                    "eta_optimize": "eta",
                    dict_keys[-1]: "ksstat"
                }).dropna()
            else:
                pre = dfg.copy()
                drop_keys = []
                opt = pd.DataFrame(columns=["r", "eta", "ksstat"]) # Empty safe fallback
            try:
                if not opt.empty:
                    _ = pre.merge(opt, on=["r", "eta"], how="outer")
            except Exception as e2:
                print(f"\nBad group: {g}")
                print(f"drop_keys: {drop_keys}")
                print(f"all keys: {dict_keys}")
                print(f"pre columns: {list(pre.columns)}")
                print(f"opt columns: {list(opt.columns)}")
                print(f"group error: {type(e2).__name__}: {e2}")
                break
        raise
    all_master_dfs.append(master_df[relevant_cols])
    
main_df = pd.concat(all_master_dfs)
main_df['best_beta'] = (main_df['best_eta'] + 1.5)/main_df['best_r'] 
main_df['best_1/beta'] = 1/main_df['best_beta']
main_df['beat_all_priors'] = (main_df['kstest_stat_best'] < np.minimum.reduce([main_df['kstest_stat_gaussian'], main_df['kstest_stat_laplace'], main_df['kstest_stat_t']])).astype(int)
main_df["best_prior"] = np.array(["GenGamma", "Gaussian", "Laplace", "Student-T", np.nan])[
                                np.nanargmin(np.array([main_df['kstest_stat_best'], 
                                                        main_df['kstest_stat_gaussian'], 
                                                        main_df['kstest_stat_laplace'], 
                                                        main_df['kstest_stat_t'], 
                                                        0.99*np.ones_like(main_df['kstest_stat_t'])]
                                                        ).T, axis=1)]
print("Main DF (before frequency):", main_df.shape)
frequency_map = pd.read_csv(os.path.join(ROOT_DIR, "transformed-data", "master-frequency-map.csv")).set_index(['dataset', 'transform', 'group'])
main_df = main_df.set_index(['dataset', 'subset', 'transform', 'group']).merge(frequency_map, left_index = True, right_index=True, how='left').reset_index()
print("Main DF (after frequency):", main_df.shape)
old_fail_cat_df_path = os.path.join(ROOT_DIR, "publication", "paper", "CSVs", 'result_categorization_sheet - combined_categories.csv')
old_fail_cat_df = pd.read_csv(old_fail_cat_df_path)
main_df = main_df.merge(old_fail_cat_df[['github_plot', 'failure_category', 'failure_type']], on='github_plot', how='left')
print("Main DF (after result categorization):", main_df.shape)

  0%|          | 0/132 [00:00<?, ?it/s]

100%|██████████| 132/132 [00:26<00:00,  5.00it/s]

Main DF (before frequency): (1176, 45)
Main DF (after frequency): (1176, 46)
Main DF (after result categorization): (1176, 48)


In [10]:
new_fail_cat_df_path = os.path.join(ROOT_DIR, "publication", "paper", "CSVs", 'new_fail_cat_df.csv')
main_df[["total_samples","dataset", "subset" , "transform" , "orientation" , "channel" , "group" , 
        "kstest_stat_best", "kstest_stat_cutoff_0.05", "beat_all_priors", "failure_category", "failure_type", "github_plot"]].to_csv(new_fail_cat_df_path)

In [13]:
save_cols = (relevant_cols +
        ['total_samples', 'beat_all_priors', 'best_prior', 'failure_category', 'failure_type'])


In [14]:
main_df[save_cols].drop(['hull'], axis=1).to_csv(save_path_without_hull)
pd.to_pickle(main_df[save_cols], save_path_with_hull)